In [729]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_percentage_error
import tensorflow as tf
from keras import layers, models, optimizers, losses, metrics
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.layers import Input, Dense, Dropout, LSTM
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, Flatten, Dense

In [730]:
df = pd.read_csv('dummy_data/cleaned_data.csv')

In [731]:
df.head()

,subject_id,hadm_id,admission_type,admission_location,insurance,language,marital_status,race,hospital_expire_flag,ccs_seq,...,hour_cos,dow_sin,dow_cos,mon_sin,mon_cos,admission_index,days_since_prev_admission,days_since_first_admission,num_prior_admissions,has_prior_admission
0,10000690,26504700,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"['CIR019', 'CIR019', 'CIR019', 'CIR003', 'CIR0...",...,8.660254e-01,-0.433884,-0.900969,1.224647e-16,-1.000000e+00,0,0.000000,0.000000,0,0
1,10000690,23280645,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"['CIR019', 'CIR019', 'RSP002', 'CIR017', 'END0...",...,2.588190e-01,0.974928,-0.222521,-8.660254e-01,-5.000000e-01,1,71.170833,75.709722,1,1
2,10000690,25860671,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"['RSP010', 'CIR019', 'RSP012', 'RSP012', 'GEN0...",...,-1.836970e-16,0.000000,1.000000,-8.660254e-01,5.000000e-01,2,39.175000,122.636111,2,1
3,10000690,26146595,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"['DIG012', 'DIG012', 'DIG012', 'GEN002', 'CIR0...",...,9.659258e-01,-0.433884,-0.900969,0.000000e+00,1.000000e+00,3,442.413194,574.870833,3,1
4,10001919,29897682,SURGICAL SAME DAY ADMISSION,PHYSICIAN REFERRAL,Private,English,MARRIED,OTHER,0,"['NEO013', 'NEO070', 'CIR033', 'DIG024', 'XXX0...",...,1.000000e+00,0.433884,-0.900969,1.000000e+00,6.123234e-17,0,0.000000,0.000000,0,0


In [733]:
# Assume df has columns: subject_id, admission_index, los, mort, ...
df = df.sort_values(['subject_id', 'admission_index'])

# Shift LOS and mortality within each patient
df['prev_los']  = df.groupby('subject_id')['admission_duration'].shift(1).fillna(0)
df['prev_mort'] = df.groupby('subject_id')['hospital_expire_flag'].shift(1).fillna(0)

In [734]:
len(NUMERIC)

15

In [735]:
# 1) Make a missingness indicator (0/1)
df["ed_duration_missing"] = df["ed_duration"].isna().astype(np.float32)

# 2) Impute a constant that is neutral + safe for later scaling
#    (constant imputation avoids leakage since it doesn't use any split statistics)
df["ed_duration"] = df["ed_duration"].fillna(0.0)

# 3) (Optional) Clean obvious errors and cap outliers
#    - negative durations -> 0
df.loc[df["ed_duration"] < 0, "ed_duration"] = 0.0
#    - cap extreme values to reduce the chance of exploding scales later
cap = df["ed_duration"].quantile(0.999)  # very high cap
df["ed_duration"] = np.clip(df["ed_duration"], 0.0, cap)

# 4) (Optional) If you plan to log-transform, make it log-safe
# df["ed_duration_log1p"] = np.log1p(df["ed_duration"])

In [736]:
CATEGORICAL = ['admission_type', 'admission_location', 'insurance', 'language', 'marital_status', 'race', 'gender']

NUMERIC = ['prev_los', 'prev_mort', 'ed_duration','age_at_admission', 'hour_sin', 'hour_cos',
                 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'days_since_prev_admission',
                 'days_since_first_admission', 'ccs_seq_len',
                 'admission_index', 'num_prior_admissions']

BINARY = ['has_prior_admission', 'is_weekend', 'is_night_admit', 'ed_duration_missing']

In [737]:
def clean_seq(x):
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except:
            return [x]
    if isinstance(x, (list, tuple, np.ndarray)):
        if len(x) == 1 and isinstance(x[0], (list, tuple, np.ndarray)):
            return list(x[0])
        return list(x)
    return [x]

df["ccs_seq"] = df["ccs_seq"].apply(clean_seq)


In [738]:
df.fillna(0, inplace=True)  # simple imputation

In [739]:
#df.fillna(0, inplace=True)  # simple imputation
df.isnull().sum()

subject_id                    0
hadm_id                       0
admission_type                0
admission_location            0
insurance                     0
language                      0
marital_status                0
race                          0
hospital_expire_flag          0
ccs_seq                       0
ccs_seq_len                   0
gender                        0
anchor_year_group             0
admission_duration            0
ed_duration                   0
age_at_admission              0
is_weekend                    0
is_night_admit                0
hour_sin                      0
hour_cos                      0
dow_sin                       0
dow_cos                       0
mon_sin                       0
mon_cos                       0
admission_index               0
days_since_prev_admission     0
days_since_first_admission    0
num_prior_admissions          0
has_prior_admission           0
prev_los                      0
prev_mort                     0
ed_durat

In [740]:
df.head()

,subject_id,hadm_id,admission_type,admission_location,insurance,language,marital_status,race,hospital_expire_flag,ccs_seq,...,mon_sin,mon_cos,admission_index,days_since_prev_admission,days_since_first_admission,num_prior_admissions,has_prior_admission,prev_los,prev_mort,ed_duration_missing
0,10000690,26504700,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"[CIR019, CIR019, CIR019, CIR003, CIR003, SYM01...",...,1.224647e-16,-1.000000e+00,0,0.000000,0.000000,0,0,0.000000,0.0,0.0
1,10000690,23280645,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"[CIR019, CIR019, RSP002, CIR017, END011, CIR01...",...,-8.660254e-01,-5.000000e-01,1,71.170833,75.709722,1,1,4.538889,0.0,0.0
2,10000690,25860671,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"[RSP010, CIR019, RSP012, RSP012, GEN002, DIG02...",...,-8.660254e-01,5.000000e-01,2,39.175000,122.636111,2,1,7.751389,0.0,0.0
3,10000690,26146595,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"[DIG012, DIG012, DIG012, GEN002, CIR019, NVS01...",...,0.000000e+00,1.000000e+00,3,442.413194,574.870833,3,1,9.821528,0.0,0.0
4,10001919,29897682,SURGICAL SAME DAY ADMISSION,PHYSICIAN REFERRAL,Private,English,MARRIED,OTHER,0,"[NEO013, NEO070, CIR033, DIG024, XXX000, XXX00...",...,1.000000e+00,6.123234e-17,0,0.000000,0.000000,0,0,0.000000,0.0,1.0


In [741]:
# Collapse to patient-level labels
patient_labels = (
    df.groupby("subject_id")["hospital_expire_flag"]
      .max()  # if any admission flagged death, mark as 1
      .reset_index()
)

X = patient_labels["subject_id"]
y = patient_labels["hospital_expire_flag"]

# Stratified split by patient_id and label
train_ids, test_ids, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ensures same proportion of outcome classes
)

# Assign admissions back to splits
train_df = df[df["subject_id"].isin(train_ids)]
test_df  = df[df["subject_id"].isin(test_ids)]

print(train_df.shape, test_df.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(9577, 32) (2251, 32)
hospital_expire_flag
0    0.94525
1    0.05475
Name: proportion, dtype: float64
hospital_expire_flag
0    0.945
1    0.055
Name: proportion, dtype: float64


In [742]:
train_df.shape

(9577, 32)

In [743]:
test_df.shape

(2251, 32)

In [744]:
### functions to encode variables ###
class CatEncoder:
    """Encodes categorical variables as integer IDs for embedding layers."""
    def __init__(self):
        self.to_id = {"<UNK>": 0}   # reserve 0 for unknown/padding
        self.from_id = ["<UNK>"]
    
    def fit(self, values: pd.Series):
        for v in values.fillna("<UNK>").astype(str).unique():
            if v not in self.to_id:
                self.to_id[v] = len(self.from_id)
                self.from_id.append(v)
        return self
    
    def transform(self, values: pd.Series) -> np.ndarray:
        vals = values.fillna("<UNK>").astype(str).values
        return np.array([self.to_id.get(v, 0) for v in vals], dtype="int32")
    
    def vocab_size(self):
        return len(self.from_id)

In [745]:
train_df = train_df.copy()
test_df  = test_df.copy()

# CATEGORICAL (embedding IDs)
for i in CATEGORICAL:
    categorical_enc = CatEncoder().fit(train_df[i])
    train_df.loc[:,i] = categorical_enc.transform(train_df[i])
    test_df.loc[:,i] = categorical_enc.transform(test_df[i])

# NUMERICAL (scaling)
scaler = StandardScaler().fit(train_df[NUMERIC])
train_df.loc[:, NUMERIC] = scaler.transform(train_df[NUMERIC])
test_df.loc[:, NUMERIC]  = scaler.transform(test_df[NUMERIC])

C:\Users\amydu\AppData\Local\Temp\ipykernel_7784\979772798.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 1.36941544  1.36941544  1.36941544 ...  0.96260707  0.09813928
 -0.8680306 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, NUMERIC] = scaler.transform(train_df[NUMERIC])
C:\Users\amydu\AppData\Local\Temp\ipykernel_7784\979772798.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.40147304 -0.28309171 -0.16471038 ... -0.28309171 -0.40147304
 -0.40147304]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, NUMERIC] = scaler.transform(train_df[NUMERIC])
C:\Users\amydu\AppData\Local\Temp\ipykernel_7784\979772798.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of

In [746]:
train_df.head()

,subject_id,hadm_id,admission_type,admission_location,insurance,language,marital_status,race,hospital_expire_flag,ccs_seq,...,mon_sin,mon_cos,admission_index,days_since_prev_admission,days_since_first_admission,num_prior_admissions,has_prior_admission,prev_los,prev_mort,ed_duration_missing
0,10000690,26504700,1,1,1,1,1,1,0,"[CIR019, CIR019, CIR019, CIR003, CIR003, SYM01...",...,0.017382,-1.399327,-0.401473,-0.394494,-0.610957,-0.401473,0,-0.477571,0.0,0.0
1,10000690,23280645,1,1,1,1,1,1,0,"[CIR019, CIR019, RSP002, CIR017, END011, CIR01...",...,-1.203064,-0.689455,-0.283092,-0.261598,-0.539271,-0.283092,1,0.266027,0.0,0.0
2,10000690,25860671,1,1,1,1,1,1,0,"[RSP010, CIR019, RSP012, RSP012, GEN002, DIG02...",...,-1.203064,0.730288,-0.164710,-0.321344,-0.494839,-0.164710,1,0.792325,0.0,0.0
3,10000690,26146595,1,1,1,1,1,1,0,"[DIG012, DIG012, DIG012, GEN002, CIR019, NVS01...",...,0.017382,1.440160,-0.046329,0.431617,-0.066639,-0.046329,1,1.131472,0.0,0.0
4,10001919,29897682,2,2,2,1,2,2,0,"[NEO013, NEO070, CIR033, DIG024, XXX000, XXX00...",...,1.426632,0.020417,-0.401473,-0.394494,-0.610957,-0.401473,0,-0.477571,0.0,1.0


In [747]:
# Explode list-like column into individual codes
train_codes = (
    train_df['ccs_seq']
    .explode()
    .dropna()                     # remove NaN
    .astype(str)                  # ensure all are strings
    .unique()
)

test_codes = (
    test_df['ccs_seq']
    .explode()
    .dropna()
    .astype(str)
    .unique()
)

# Build vocab from TRAIN only
unique_codes = sorted(set(train_codes))

# Add PAD and UNK
code2idx = {code: idx+2 for idx, code in enumerate(unique_codes)}
code2idx["<PAD>"] = 0 ##for padding up to max length
code2idx["<UNK>"] = 1 ##for any unknown codes in test set

#convert each code to integer ID, use UNK for any unknown codes
#outputs a list of integer IDs for each admission diagnosis sequence
def encode_sequence(seq, mapping):
    return [mapping.get(str(c), mapping["<UNK>"]) for c in seq if c is not None]

#longest sequence in data
max_len = 568

#convert ccs seq to integers
train_df["ccs_encoded"] = train_df["ccs_seq"].apply(lambda x: encode_sequence(x, code2idx))
#pad sequences to max length with 0s
train_df["diag_seq"] = pad_sequences(train_df["ccs_encoded"], maxlen=max_len, padding="post").tolist()

#apply to test df
test_df["ccs_encoded"] = test_df["ccs_seq"].apply(lambda x: encode_sequence(x, code2idx))
test_df["diag_seq"] = pad_sequences(test_df["ccs_encoded"], maxlen=max_len, padding="post").tolist()


In [748]:
#drop old columns
train_df.drop(columns=["ccs_seq", "ccs_encoded"], inplace=True)
test_df.drop(columns=["ccs_seq", "ccs_encoded"], inplace=True)

In [749]:
def to_array_safe(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.empty((0,), dtype=np.float32)
    if isinstance(x, np.ndarray):
        return x.astype(np.float32)
    if isinstance(x, (list, tuple)):
        return np.asarray(x, dtype=np.float32)
    # if a scalar slipped in, wrap it
    try:
        return np.asarray([x], dtype=np.float32)
    except Exception:
        return np.empty((0,), dtype=np.float32)

train_df["diag_seq"] = train_df["diag_seq"].apply(to_array_safe)
test_df["diag_seq"]  = test_df["diag_seq"].apply(to_array_safe)

In [750]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Any

def _pad_3d_rows(rows: List[np.ndarray], T: int, inner_shape: Tuple[int, ...], pad_value: float = 0.0) -> np.ndarray:
    """
    Rows are arrays with shape inner_shape (e.g., (E,) or (L,E)).
    Return (T, *inner_shape) padded with pad_value.
    """
    out = np.full((T,) + inner_shape, pad_value, dtype=np.float32)
    L = min(len(rows), T)
    for t in range(L):
        a = rows[t]
        slices = tuple(slice(0, min(a.shape[d], inner_shape[d])) for d in range(len(inner_shape)))
        out[(t,)+slices] = a[slices]
    return out

def build_patient_tensors(
    df: pd.DataFrame,
    patient_col: str,
    order_col: str,
    cat_cols: List[str],
    num_cols: List[str],
    binary_cols: List[str],
    diag_col: str,
    max_T: int = None,
    pad_value: float = 0.0,
) -> Dict[str, Any]:
    """
    Returns:
      - X_cat:        (N, T, Kc)  int32
      - X_num:        (N, T, Kn)  float32
      - X_diag:       (N, T, *diag_shape) float32
      - mask_time:    (N, T)  float32 (1=real admission, 0=pad)
      - patients:     (N,) list of patient ids (order matches tensors)
      - hadm_ids:     (N, T) object array of admission IDs (optional, if present in df)
    """
    # Ensure lists
    cat_cols = list(cat_cols)
    num_cols = list(num_cols)   
    binary_cols = list(binary_cols)

    # Ascending order per patient
    df_sorted = df.sort_values([patient_col, order_col]).copy()

    # Infer diagnosis embedding shape from first non-null row
    first_non_null = next((v for v in df_sorted[diag_col].values if isinstance(v, np.ndarray)), None)
    if first_non_null is None:
        raise ValueError(f"No numpy arrays found in column '{diag_col}'.")
    diag_shape = first_non_null.shape  # e.g., (E,) or (L,E)

    # Group by patient
    groups = list(df_sorted.groupby(patient_col, sort=False))

    # Compute T
    lengths = [len(g) for _, g in groups]
    T = max(lengths) if max_T is None else int(max_T)   # always use max_T when given
    N = len(groups)

    Kc = len(cat_cols)
    Kn = len(num_cols)
    Kb = len(binary_cols)

    X_cat  = np.zeros((N, T, Kc), dtype="int32")
    X_num  = np.zeros((N, T, Kn), dtype="float32")
    X_bin  = np.zeros((N, T, Kb), dtype="float32")
    X_diag = np.full((N, T) + diag_shape, pad_value, dtype="float32")
    mask   = np.zeros((N, T), dtype="float32")
    hadm   = np.empty((N, T), dtype=object) if "hadm_id" in df.columns else None

    patients = []

    for i, (pid, g) in enumerate(groups):
        patients.append(pid)

        # Within-patient arrays
        cat_rows = g[cat_cols].to_numpy(dtype="int32") if Kc > 0 else None  # (len_g, Kc)
        num_rows = g[num_cols].to_numpy(dtype="float32") if Kn > 0 else None
        bin_rows = g[binary_cols].to_numpy(dtype="float32") if Kb > 0 else None
        diag_rows = g[diag_col].tolist()  # list of np arrays

        L = min(len(g), T)

        # Fill mask
        mask[i, :L] = 1.0

        # Fill hadm ids if available
        if hadm is not None:
            vals = g["hadm_id"].tolist()
            hadm[i, :L] = vals[:L]

        # Categorical / numeric
        if Kc > 0:
            # pad (T, Kc)
            tmp = np.zeros((T, Kc), dtype="int32")
            if L > 0:
                tmp[:L, :] = cat_rows[:L, :]
            X_cat[i] = tmp

        if Kn > 0:
            tmp = np.zeros((T, Kn), dtype="float32")
            if L > 0:
                tmp[:L, :] = num_rows[:L, :]
            X_num[i] = tmp
        
        if Kb > 0:
            tmp = np.zeros((T, Kb), dtype="float32")
            if L > 0:
                tmp[:L, :] = bin_rows[:L, :]
            X_bin[i] = tmp

        # Diagnosis embedding per admission (shape diag_shape)
        X_diag[i] = _pad_3d_rows(diag_rows, T, diag_shape, pad_value=pad_value)

    out = {
        "X_cat": X_cat,            # (N, T, Kc)
        "X_num": X_num,            # (N, T, Kn)
        "X_bin": X_bin,            # (N, T, Kb)
        "X_diag": X_diag,          # (N, T, *diag_shape)
        "mask_time": mask,         # (N, T)
        "patients": np.array(patients),
    }
    if hadm is not None:
        out["hadm_ids"] = hadm
    return out


In [751]:
train_tensor = build_patient_tensors(
    train_df,
    patient_col="subject_id",
    order_col="admission_index",
    cat_cols=CATEGORICAL,
    num_cols=NUMERIC,
    binary_cols=BINARY,
    diag_col="diag_seq",
    max_T=14,
    pad_value=0.0,
)

In [752]:
X_cat_train     = train_tensor["X_cat"]       # (N, T, Kc)
X_num_train     = train_tensor["X_num"]       # (N, T, Kn)
X_bin_train     = train_tensor["X_bin"]       # (N, T, Kb)
X_diag_train    = train_tensor["X_diag"]      # (N, T, E) or (N, T, L, E)
mask_time_train = train_tensor["mask_time"]   # (N, T)
patients_train  = train_tensor["patients"]    # (N,)

In [753]:
test_tensor = build_patient_tensors(
    test_df,
    patient_col="subject_id",
    order_col="admission_index",
    cat_cols=CATEGORICAL,
    binary_cols=BINARY,
    num_cols=NUMERIC,
    diag_col="diag_seq",
    max_T=14,
    pad_value=0.0,
)

X_cat_test      = test_tensor["X_cat"]        # (N, T, Kc)
X_num_test      = test_tensor["X_num"]        # (N, T, Kn)
X_diag_test     = test_tensor["X_diag"]       # (N, T, *diag_shape)
X_bin_test      = test_tensor["X_bin"]        # (N, T, Kb)
mask_time_test  = test_tensor["mask_time"]    # (N, T)
patients_test   = test_tensor["patients"]     # (N,)

In [759]:
X_bin_test.shape

(1000, 14, 4)

In [755]:
class_weight

{'mortality': {0: np.float64(0.5289605924358635),
  1: np.float64(9.132420091324201)},
 'los': 1.0}

In [756]:
import keras
keras.backend.clear_session()


In [760]:
import tensorflow as tf
from keras import layers, Model, metrics, callbacks, backend as K
K.clear_session()

# shapes from your tensors
T, Kc, Kn, Kb, E = X_cat_train.shape[1], X_cat_train.shape[2], X_num_train.shape[2], X_bin_train.shape[2], X_diag_train.shape[2]

# inputs (unique names)
inp_cat  = layers.Input(shape=(T, Kc), dtype="int32",   name="cat_in")
inp_num  = layers.Input(shape=(T, Kn), dtype="float32", name="num_in")
inp_bin  = layers.Input(shape=(T, Kb), dtype="float32", name="bin_in")
inp_diag = layers.Input(shape=(T, E),  dtype="float32", name="diag_in")
inp_mask = layers.Input(shape=(T,),    dtype="float32", name="mask_in")

# fuse
cat_f  = layers.Lambda(lambda x: tf.cast(x, tf.float32), name="cat_cast")(inp_cat)
fusion = layers.Concatenate(name="visit_fusion")([cat_f, inp_num, inp_bin, inp_diag])

# mask and encoder (return_sequences=False -> final state only)
mask_bool = layers.Lambda(lambda m: tf.cast(m > 0.5, tf.bool), name="mask_bool")(inp_mask)
h_seq = layers.LSTM(128, return_sequences=True, name="lstm")(fusion, mask=mask_bool)  # (B, 256)

#attention layer
# scalar scores per timestep
score = layers.Dense(1, name="attn_score")(h_seq)     # (B, T, 1)

# mask pads by adding -inf to their scores BEFORE softmax
mask_f = layers.Lambda(lambda m: tf.cast(m, tf.float32), name="mask_float")(mask_bool)  # (B, T)
masked_score = layers.Lambda(
    lambda xs: xs[0] + (1.0 - xs[1])[..., None] * (-1e9),
    name="attn_mask_scores"
)([score, mask_f])                                     # (B, T, 1)

# softmax over time
weights = layers.Softmax(axis=1, name="attn_weights")(masked_score)  # (B, T, 1)

# weighted sum of hidden states -> context vector
weighted = layers.Multiply(name="attn_weighted")([weights, h_seq])   # (B, T, 256)
context = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1),
                        name="attn_context")(weighted)               # (B, 256)

# trunk + heads
z = layers.Dense(128, activation="relu", name="trunk_dense")(context)
z = layers.Dropout(0.2, name="trunk_drop")(z)


from keras.initializers import Constant
pos = y_train_mort.mean()  # ~0.055
init_bias = np.log(pos/(1. - pos))

mortality = layers.Dense(1, activation="sigmoid", name="mortality",
                         bias_initializer=Constant(init_bias))(z)
los       = layers.Dense(1, activation="linear",  name="los")(z)

model = Model(inputs=[inp_cat, inp_num, inp_bin, inp_diag, inp_mask],
              outputs={"mortality": mortality, "los": los})

# compile with multiple losses and metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss={"mortality": "binary_crossentropy", "los": "mse"},
    metrics={
        "mortality": [
            metrics.AUC(name="auc"),
            metrics.AUC(name="pr_auc", curve="PR"),
            metrics.Precision(name="precision"),
            metrics.Recall(name="recall"),
            metrics.BinaryAccuracy(name="accuracy"),
        ],
        "los": [metrics.MeanAbsoluteError(name="mae"), metrics.MeanSquaredError(name="mse")],
    },
    loss_weights={"mortality": 3.0, "los": 1.0},   # tune if needed
)

model.summary()

# counts
p = int((y_train_mort == 1).sum())
n = int((y_train_mort == 0).sum())
w_pos = n / max(p, 1)         # e.g. ~17–20 for ~5–6% positives
w_neg = 1.0

# per-sample weights for the mortality head
sw_mort = np.where(y_train_mort == 1, w_pos, w_neg).astype("float32")
# keep LOS unweighted (or set your own)
sw_los  = np.ones_like(y_train_los, dtype="float32")

#es = callbacks.EarlyStopping(monitor="val_mortality_auc", mode="max", patience=5, restore_best_weights=True)

print("Xc_tr shape:", Xc_tr.shape)
print("Xn_tr shape:", Xn_tr.shape)
print("Xd_tr shape:", Xd_tr.shape)
print("M_tr shape :", M_tr.shape)
print("Xb_tr shape:", Xb_tr.shape)

print("y_train_mort shape:", y_train_mort.shape,
      "positive count:", (y_train_mort==1).sum(),
      "negative count:", (y_train_mort==0).sum())

print("y_train_los shape:", y_train_los.shape)

# most important: check sample weights
print("sw_mort shape:", sw_mort.shape, "unique values:", np.unique(sw_mort)[:10])
print("sw_los shape:", sw_los.shape, "unique values:", np.unique(sw_los)[:10])                   

train_inputs = {
    "cat_in":  X_cat_train,
    "num_in":  X_num_train,
    "bin_in":  X_bin_train,
    "diag_in": X_diag_train,
    "mask_in": mask_time_train.astype("float32"),
}
val_inputs = {
    "cat_in":  X_cat_test,
    "num_in":  X_num_test,
    "bin_in":  X_bin_test,
    "diag_in": X_diag_test,
    "mask_in": mask_time_test.astype("float32"),
}

hist = model.fit(
    x=train_inputs,
    y={"mortality": y_train_mort, "los": y_train_los},
    validation_data=(val_inputs, {"mortality": y_test_mort, "los": y_test_los}),
    epochs=50,
    batch_size=64,
    #callbacks=[es],
    #sample_weight={"mortality": sw_mort, "los": sw_los},
    verbose=1
)

# -------- 4) Evaluation --------
eval_out = model.evaluate(val_inputs, {"mortality": y_test_mort, "los": y_test_los}, verbose=0)
# Keras returns a flat list: [total_loss, mort_loss, los_loss, mort_auc, mort_pr_auc, ... , los_mae, los_mse]
print("Eval:", eval_out)

# Extra: thresholded mort metrics + LOS MAE in original units (if you log-transformed, invert here)
y_prob = model.predict(val_inputs, batch_size=256, verbose=0)["mortality"].ravel()

from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
y_hat = (y_prob >= 0.5).astype(int)

print("Mortality — ROC AUC:", roc_auc_score(y_test_mort, y_prob))
print("Mortality — PR  AUC:", average_precision_score(y_test_mort, y_prob))
print("Mortality — Precision:", precision_score(y_test_mort, y_hat, zero_division=0))
print("Mortality — Recall   :", recall_score(y_test_mort, y_hat, zero_division=0))
print("Mortality — F1       :", f1_score(y_test_mort, y_hat, zero_division=0))

# LOS predictions
y_pred_los = model.predict(val_inputs, batch_size=256, verbose=0)["los"].ravel()

# If you used log1p, invert:
# y_pred_los = np.expm1(y_pred_los)
# y_test_los = np.expm1(y_test_los)

mae = np.mean(np.abs(y_pred_los - y_test_los))
mse = np.mean((y_pred_los - y_test_los)**2)
print("LOS — MAE:", mae, "   MSE:", mse)


c:\Users\amydu\anaconda3\envs\tf311\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'attn_mask_scores' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cat_in (InputLayer) │ (None, 14, 7)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_cast (Lambda)   │ (None, 14, 7)     │          0 │ cat_in[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_in (InputLayer) │ (None, 14, 15)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bin_in (InputLayer) │ (None, 14, 4)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diag_in             │ (None, 14, 568)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask_in             │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ visit_fusion        │ (None, 14, 594)   │          0 │ cat_cast[0][0],   │
│ (Concatenate)       │                   │            │ num_in[0][0],     │
│                     │                   │            │ bin_in[0][0],     │
│                     │                   │            │ diag_in[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask_bool (Lambda)  │ (None, 14)        │          0 │ mask_in[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 14, 128)   │    370,176 │ visit_fusion[0][… │
│                     │                   │            │ mask_bool[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_score (Dense)  │ (None, 14, 1)     │        129 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask_float (Lambda) │ (None, 14)        │          0 │ mask_bool[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_mask_scores    │ (None, 14, 1)     │          0 │ attn_score[0][0], │
│ (Lambda)            │                   │            │ mask_float[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_weights        │ (None, 14, 1)     │          0 │ attn_mask_scores… │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_weighted       │ (None, 14, 128)   │          0 │ attn_weights[0][… │
│ (Multiply)          │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_context        │ (None, 128)       │          0 │ attn_weighted[0]… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ trunk_dense (Dense) │ (None, 128)       │     16,512 │ attn_context[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ trunk_drop          │ (None, 128)       │          0 │ trunk_dense[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ los (Dense)         │ (None, 1)         │        129 │ trunk_drop[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mortality (Dense)   │ (None, 1)         │        129 │ trunk_drop[0][0]  │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 387,075 (1.48 MB)

 Trainable params: 387,075 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

Xc_tr shape: (4000, 14, 7)
Xn_tr shape: (4000, 14, 15)
Xd_tr shape: (4000, 14, 568)
M_tr shape : (4000, 14)
Xb_tr shape: (4000, 14, 3)
y_train_mort shape: (4000,) positive count: 219 negative count: 3781
y_train_los shape: (4000,)
sw_mort shape: (4000,) unique values: [ 1.       17.264841]
sw_los shape: (4000,) unique values: [1.]
Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - los_loss: 2.9564 - los_mae: 1.4699 - los_mse: 2.9648 - loss: 3.6097 - mortality_accuracy: 0.9452 - mortality_auc: 0.4965 - mortality_loss: 0.2153 - mortality_pr_auc: 0.0561 - mortality_precision: 0.0000e+00 - mortality_recall: 0.0000e+00 - val_los_loss: 1.5883 - val_los_mae: 0.9998 - val_los_mse: 1.5847 - val_loss: 2.2240 - val_mortality_accuracy: 0.9450 - val_mortality_auc: 0.5309 - val_mortality_loss: 0.2163 - val_mortality_pr_auc: 0.0600 - val_mortality_precision: 0.0000e+00 - val_mortality_recall: 0.0000e+00
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - los_loss: 1.2251 - los_mae: 0.8477 - los_m

In [761]:
from sklearn.metrics import precision_recall_curve

y_prob = mort_model.predict(val_inputs, 256).ravel()
prec, rec, th = precision_recall_curve(y_test_mort, y_prob)

# pick threshold that maximizes F1
f1 = 2*prec*rec / (prec+rec+1e-12)
best_idx = f1.argmax()
print("Best threshold:", th[best_idx], "Precision:", prec[best_idx], "Recall:", rec[best_idx], "F1:", f1[best_idx])


c:\Users\amydu\anaconda3\envs\tf311\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'attn_mask_scores' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


InvalidArgumentError: Graph execution error:

Detected at node MortalityOnly_1/lstm_1/while/body/_1/MortalityOnly_1/lstm_1/while/lstm_cell_1/MatMul defined at (most recent call last):
<stack traces unavailable>
Matrix size-incompatible: In[0]: [256,594], In[1]: [593,512]
	 [[{{node MortalityOnly_1/lstm_1/while/body/_1/MortalityOnly_1/lstm_1/while/lstm_cell_1/MatMul}}]] [Op:__inference_one_step_on_data_distributed_1715371]

In [691]:
from sklearn.metrics import roc_auc_score, average_precision_score
print("sklearn ROC AUC:", roc_auc_score(y_test_mort, y_prob))
print("sklearn PR  AUC:", average_precision_score(y_test_mort, y_prob))

sklearn ROC AUC: 0.5
sklearn PR  AUC: 0.055


In [696]:
import numpy as np
import tensorflow as tf
from keras import layers, Model, metrics, callbacks, backend as K
from keras.initializers import Constant

# -------------------------
# 0) Common shapes/datasets
# -------------------------
# Assumes you already have:
# X_cat_train, X_num_train, X_bin_train, X_diag_train, mask_time_train
# X_cat_test,  X_num_test,  X_bin_test,  X_diag_test,  mask_time_test
# y_train_mort, y_test_mort  (0/1)
# y_train_los,  y_test_los   (float)
T  = X_cat_train.shape[1]
Kc = X_cat_train.shape[2]
Kn = X_num_train.shape[2]
Kb = X_bin_train.shape[2]
E  = X_diag_train.shape[2]

# -------------------------
# 1) Reusable attention trunk
# -------------------------
def make_inputs():
    inp_cat  = layers.Input(shape=(T, Kc), dtype="int32",   name="cat_in")
    inp_num  = layers.Input(shape=(T, Kn), dtype="float32", name="num_in")
    inp_bin  = layers.Input(shape=(T, Kb), dtype="float32", name="bin_in")
    inp_diag = layers.Input(shape=(T, E),  dtype="float32", name="diag_in")
    inp_mask = layers.Input(shape=(T,),    dtype="float32", name="mask_in")
    return inp_cat, inp_num, inp_bin, inp_diag, inp_mask

def attention_trunk(inp_cat, inp_num, inp_bin, inp_diag, inp_mask,
                    lstm_units=128, dropout=0.2):
    # cast + fuse
    cat_f  = layers.Lambda(lambda x: tf.cast(x, tf.float32), name="cat_cast")(inp_cat)
    fusion = layers.Concatenate(name="visit_fusion")([cat_f, inp_num, inp_bin, inp_diag])  # (B,T, F)

    # mask -> bool
    mask_bool = layers.Lambda(lambda m: tf.cast(m > 0.5, tf.bool), name="mask_bool")(inp_mask)

    # sequence encoder
    h_seq = layers.LSTM(lstm_units, return_sequences=True, name="lstm")(fusion, mask=mask_bool)  # (B,T,U)

    # attention (mask-aware)
    score = layers.Dense(1, name="attn_score")(h_seq)                           # (B,T,1)
    mask_f = layers.Lambda(lambda m: tf.cast(m, tf.float32), name="mask_float")(mask_bool)  # (B,T)
    masked_score = layers.Lambda(
        lambda xs: xs[0] + (1.0 - xs[1])[..., None] * (-1e9),
        name="attn_mask_scores"
    )([score, mask_f])                                                          # (B,T,1)
    weights  = layers.Softmax(axis=1, name="attn_weights")(masked_score)        # (B,T,1)
    weighted = layers.Multiply(name="attn_weighted")([weights, h_seq])          # (B,T,U)
    context  = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1),
                             name="attn_context")(weighted)                      # (B,U)

    # small trunk
    z = layers.Dense(128, activation="relu", name="trunk_dense")(context)
    z = layers.Dropout(dropout, name="trunk_drop")(z)
    return z

# ------------------------------------
# 2) Mortality-only classification model
# ------------------------------------
K.clear_session()
inp_cat, inp_num, inp_bin, inp_diag, inp_mask = make_inputs()
z = attention_trunk(inp_cat, inp_num, inp_bin, inp_diag, inp_mask, lstm_units=128, dropout=0.2)

# Bias init to match prevalence
p = float(np.mean(y_train_mort))
bias_init = Constant(np.log(p/(1.0 - p + 1e-12)))
mort_out = layers.Dense(1, activation="sigmoid", bias_initializer=bias_init, name="mortality")(z)

mort_model = Model(
    inputs=[inp_cat, inp_num, inp_bin, inp_diag, inp_mask],
    outputs=mort_out,
    name="MortalityOnly"
)

mort_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        metrics.AUC(name="roc_auc", curve="ROC", num_thresholds=1000),
        metrics.AUC(name="pr_auc", curve="PR", num_thresholds=1000),
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.BinaryAccuracy(name="accuracy"),
    ],
)

train_mort_inputs = {
    "cat_in":  X_cat_train,
    "num_in":  X_num_train,
    "bin_in":  X_bin_train,
    "diag_in": X_diag_train,
    "mask_in": mask_time_train.astype("float32"),
}
val_mort_inputs = {
    "cat_in":  X_cat_test,
    "num_in":  X_num_test,
    "bin_in":  X_bin_test,
    "diag_in": X_diag_test,
    "mask_in": mask_time_test.astype("float32"),
}

# (optional) lighter class weight to start
# # w_pos = 5.0
# sw_mort = np.where(y_train_mort==1, w_pos, 1.0).astype("float32")
# hist_mort = mort_model.fit(train_mort_inputs, y_train_mort,
#                            validation_data=(val_mort_inputs, y_test_mort),
#                            epochs=25, batch_size=64, sample_weight=sw_mort, verbose=1)

hist_mort = mort_model.fit(
    train_mort_inputs, y_train_mort,
    validation_data=(val_mort_inputs, y_test_mort),
    epochs=25, batch_size=64, verbose=1
)

# Evaluate mortality model
mort_eval = mort_model.evaluate(val_mort_inputs, y_test_mort, verbose=0)
print("Mortality eval:", dict(zip(mort_model.metrics_names, mort_eval)))

y_prob_mort = mort_model.predict(val_mort_inputs, batch_size=256, verbose=0).ravel()
print("Mort preds std/min/max:", np.std(y_prob_mort), y_prob_mort.min(), y_prob_mort.max())

# -------------------------------
# 3) LOS-only regression model
# -------------------------------
K.clear_session()
inp_cat, inp_num, inp_bin, inp_diag, inp_mask = make_inputs()
z = attention_trunk(inp_cat, inp_num, inp_bin, inp_diag, inp_mask, lstm_units=128, dropout=0.2)
los_out = layers.Dense(1, activation="linear", name="los")(z)

los_model = Model(
    inputs=[inp_cat, inp_num, inp_bin, inp_diag, inp_mask],
    outputs=los_out,
    name="LOSOnly"
)

los_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=[metrics.MeanAbsoluteError(name="mae"), metrics.MeanSquaredError(name="mse")],
)

train_los_inputs = {
    "cat_in":  X_cat_train,
    "num_in":  X_num_train,
    "bin_in":  X_bin_train,
    "diag_in": X_diag_train,
    "mask_in": mask_time_train.astype("float32"),
}
val_los_inputs = {
    "cat_in":  X_cat_test,
    "num_in":  X_num_test,
    "bin_in":  X_bin_test,
    "diag_in": X_diag_test,
    "mask_in": mask_time_test.astype("float32"),
}

hist_los = los_model.fit(
    train_los_inputs, y_train_los,
    validation_data=(val_los_inputs, y_test_los),
    epochs=25, batch_size=64, verbose=1
)

# Evaluate LOS model
los_eval = los_model.evaluate(val_los_inputs, y_test_los, verbose=0)
print("LOS eval:", dict(zip(los_model.metrics_names, los_eval)))

y_pred_los = los_model.predict(val_los_inputs, batch_size=256, verbose=0).ravel()
print("LOS preds std/min/max:", np.std(y_pred_los), y_pred_los.min(), y_pred_los.max())


c:\Users\amydu\anaconda3\envs\tf311\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'attn_mask_scores' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 49ms/step - accuracy: 0.9452 - loss: 0.2124 - pr_auc: 0.0550 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.4996 - val_accuracy: 0.9450 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000
Epoch 2/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9452 - loss: 0.2123 - pr_auc: 0.0547 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.5000 - val_accuracy: 0.9450 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000
Epoch 3/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9452 - loss: 0.2123 - pr_auc: 0.0547 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.5000 - val_accuracy: 0.9450 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000
Epoch 4/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.9452 - loss: 0.2123 - pr_a

In [697]:
# A. check prediction spread
y_prob = mort_model.predict(val_mort_inputs, batch_size=256, verbose=0).ravel()
print("mort prob std/min/max:", np.std(y_prob), y_prob.min(), y_prob.max())

# B. sklearn reference AUCs
from sklearn.metrics import roc_auc_score, average_precision_score
print("sklearn ROC AUC:", roc_auc_score(y_test_mort, y_prob))
print("sklearn PR  AUC:", average_precision_score(y_test_mort, y_prob))

# C. label distribution sanity
print("train label counts:", np.unique(y_train_mort, return_counts=True))
print("val   label counts:", np.unique(y_test_mort,  return_counts=True))

# D. make sure your dict keys match the model inputs **exactly**
print("model.input_names:", mort_model.input_names)


mort prob std/min/max: 1.1175871e-08 0.054657247 0.054657247
sklearn ROC AUC: 0.5
sklearn PR  AUC: 0.055
train label counts: (array([0, 1], dtype=int32), array([3781,  219]))
val   label counts: (array([0, 1], dtype=int32), array([945,  55]))


AttributeError: 'Functional' object has no attribute 'input_names'

In [698]:
# List input tensors (name, shape, dtype)
for t in mort_model.inputs:
    print("INPUT:", t.name, t.shape, t.dtype)

# Get the base names you must use as dict keys (strip the ':0')
input_base_names = [t.name.split(":")[0] for t in mort_model.inputs]
print("Base input names:", input_base_names)

# If you're feeding a dict, its keys must equal these names:
print("Your dict keys:", list(val_mort_inputs.keys()))
assert set(val_mort_inputs.keys()) == set(input_base_names), "Dict keys don't match model inputs!"


INPUT: cat_in (None, 14, 7) int32
INPUT: num_in (None, 14, 15) float32
INPUT: bin_in (None, 14, 3) float32
INPUT: diag_in (None, 14, 568) float32
INPUT: mask_in (None, 14) float32
Base input names: ['cat_in', 'num_in', 'bin_in', 'diag_in', 'mask_in']
Your dict keys: ['cat_in', 'num_in', 'bin_in', 'diag_in', 'mask_in']


In [699]:
import tensorflow as tf
from keras import layers, Model, metrics, backend as K
from keras.initializers import Constant
K.clear_session()

# inputs (use the same shapes you used before)
inp_cat  = layers.Input(shape=(T, Kc), dtype="int32",   name="cat_in")
inp_num  = layers.Input(shape=(T, Kn), dtype="float32", name="num_in")
inp_bin  = layers.Input(shape=(T, Kb), dtype="float32", name="bin_in")
inp_diag = layers.Input(shape=(T, E),  dtype="float32", name="diag_in")
inp_mask = layers.Input(shape=(T,),    dtype="float32", name="mask_in")

# cast + fuse
cat_f  = layers.Lambda(lambda x: tf.cast(x, tf.float32), name="cat_cast")(inp_cat)
fusion = layers.Concatenate(name="visit_fusion")([cat_f, inp_num, inp_bin, inp_diag])   # (B,T,F)

# masked mean over time: sum(x*mask)/sum(mask)
mask_exp = layers.Lambda(lambda m: tf.expand_dims(m, -1), name="mask_expand")(inp_mask) # (B,T,1)
num = layers.Multiply(name="apply_mask")([fusion, mask_exp])                              # (B,T,F)
sum_num = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1), name="sum_num")(num)         # (B,F)
sum_den = layers.Lambda(lambda m: tf.reduce_sum(m, axis=1, keepdims=False)+1e-6, name="sum_den")(inp_mask)  # (B,)
den_exp = layers.Lambda(lambda d: tf.expand_dims(d, -1), name="den_expand")(sum_den)     # (B,1)
pooled  = layers.Lambda(lambda xs: xs[0]/xs[1], name="masked_mean")([sum_num, den_exp])  # (B,F)

# small MLP
h = layers.Dense(256, activation="relu")(pooled)
h = layers.Dropout(0.3)(h)
h = layers.Dense(128, activation="relu")(h)
h = layers.Dropout(0.3)(h)

# prevalence-matched bias
p = float(np.mean(y_train_mort))
bias_init = Constant(np.log(p/(1-p + 1e-12)))
out = layers.Dense(1, activation="sigmoid", bias_initializer=bias_init, name="mortality")(h)

mort_baseline = Model([inp_cat, inp_num, inp_bin, inp_diag, inp_mask], out, name="MortalityBaseline")

mort_baseline.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[metrics.AUC(name="roc_auc", curve="ROC", num_thresholds=1000),
             metrics.AUC(name="pr_auc",  curve="PR",  num_thresholds=1000),
             metrics.BinaryAccuracy(name="accuracy")]
)

train_inputs = {
    "cat_in":  X_cat_train,
    "num_in":  X_num_train,
    "bin_in":  X_bin_train,
    "diag_in": X_diag_train,
    "mask_in": mask_time_train.astype("float32"),
}
val_inputs = {
    "cat_in":  X_cat_test,
    "num_in":  X_num_test,
    "bin_in":  X_bin_test,
    "diag_in": X_diag_test,
    "mask_in": mask_time_test.astype("float32"),
}

hist = mort_baseline.fit(train_inputs, y_train_mort,
                         validation_data=(val_inputs, y_test_mort),
                         epochs=10, batch_size=64, verbose=1)

y_prob = mort_baseline.predict(val_inputs, batch_size=256, verbose=0).ravel()
from sklearn.metrics import roc_auc_score
print("Baseline sklearn ROC AUC:", roc_auc_score(y_test_mort, y_prob))
print("Pred std/min/max:", np.std(y_prob), y_prob.min(), y_prob.max())


Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9442 - loss: 0.3128 - pr_auc: 0.0570 - roc_auc: 0.5012 - val_accuracy: 0.9450 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9452 - loss: 0.2123 - pr_auc: 0.0514 - roc_auc: 0.4794 - val_accuracy: 0.9450 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9452 - loss: 0.2123 - pr_auc: 0.0561 - roc_auc: 0.5105 - val_accuracy: 0.9450 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9452 - loss: 0.2123 - pr_auc: 0.0515 - roc_auc: 0.4871 - val_accuracy: 0.9450 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9452 - loss: 0.2123 - pr_auc: 0.0536 - roc_auc: 0.4971 - val_accuracy: 0.9450 - val_loss: 0.2130 - val_pr_auc: 0.0550 -

In [700]:
K.clear_session()
inp_cat  = layers.Input(shape=(T, Kc), dtype="int32",   name="cat_in")
inp_num  = layers.Input(shape=(T, Kn), dtype="float32", name="num_in")
inp_bin  = layers.Input(shape=(T, Kb), dtype="float32", name="bin_in")
inp_diag = layers.Input(shape=(T, E),  dtype="float32", name="diag_in")
inp_mask = layers.Input(shape=(T,),    dtype="float32", name="mask_in")

cat_f  = layers.Lambda(lambda x: tf.cast(x, tf.float32))(inp_cat)
fusion = layers.Concatenate()([cat_f, inp_num, inp_bin, inp_diag])

mask_bool = layers.Lambda(lambda m: tf.cast(m > 0.5, tf.bool))(inp_mask)
h = layers.LSTM(128, return_sequences=False)(fusion, mask=mask_bool)

h = layers.Dense(128, activation="relu")(h)
h = layers.Dropout(0.3)(h)

p = float(np.mean(y_train_mort))
bias_init = Constant(np.log(p/(1-p + 1e-12)))
out = layers.Dense(1, activation="sigmoid", bias_initializer=bias_init, name="mortality")(h)

mort_simple = Model([inp_cat, inp_num, inp_bin, inp_diag, inp_mask], out)
mort_simple.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                    loss="binary_crossentropy",
                    metrics=[metrics.AUC(name="roc_auc", curve="ROC"),
                             metrics.AUC(name="pr_auc",  curve="PR")])
mort_simple.fit(train_inputs, y_train_mort,
                validation_data=(val_inputs, y_test_mort),
                epochs=10, batch_size=64, verbose=1)


Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - loss: 0.2124 - pr_auc: 0.0542 - roc_auc: 0.4994 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.2123 - pr_auc: 0.0547 - roc_auc: 0.5000 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.2123 - pr_auc: 0.0547 - roc_auc: 0.5000 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.2123 - pr_auc: 0.0547 - roc_auc: 0.5000 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.2123 - pr_auc: 0.0547 - roc_auc: 0.5000 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.2123 - pr_auc: 0.0547 - roc_auc: 0.5000 - val_loss: 0.2130 - val_pr_auc: 0.0550 - val_roc_auc: 0.5000
Epoch 7/10
63/63 ━━━━━

In [702]:
# (A) list exact input names you're feeding
for t in mort_model.inputs:
    print("INPUT ->", t.name, t.shape, t.dtype)

# (B) build a probe to grab the penultimate tensor (before the Dense(1))
penult = mort_model.layers[-2].output  # the layer before "mortality"
probe  = tf.keras.Model(mort_model.inputs, penult)
H_val  = probe.predict(val_mort_inputs, batch_size=256, verbose=0)

print("penult std/min/max:", H_val.std(), H_val.min(), H_val.max())

INPUT -> cat_in (None, 14, 7) int32
INPUT -> num_in (None, 14, 15) float32
INPUT -> bin_in (None, 14, 3) float32
INPUT -> diag_in (None, 14, 568) float32
INPUT -> mask_in (None, 14) float32


c:\Users\amydu\anaconda3\envs\tf311\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'attn_mask_scores' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


penult std/min/max: 0.0 0.0 0.0


In [703]:
# --- Build probes for key tensors ---
get = mort_model.get_layer

# 1) LSTM sequence output (before attention)
probe_lstm = tf.keras.Model(mort_model.inputs, get("lstm").output)          # (B,T,U)

# 2) Attention score (before masking)
probe_score = tf.keras.Model(mort_model.inputs, get("attn_score").output)   # (B,T,1)

# 3) Attention weights (after masking + softmax)
probe_w = tf.keras.Model(mort_model.inputs, get("attn_weights").output)     # (B,T,1)

# 4) Context (weighted sum)
probe_ctx = tf.keras.Model(mort_model.inputs, get("attn_context").output)   # (B,U)

# 5) Trunk dense (after ReLU, the "penult" you saw is likely dropout output)
probe_trunk = tf.keras.Model(mort_model.inputs, get("trunk_dense").output)  # (B,128)

# --- Run them on your val set ---
X = val_mort_inputs
H   = probe_lstm.predict(X, 256);      print("LSTM  std/min/max:", H.std(),   H.min(),   H.max())
S   = probe_score.predict(X, 256);     print("Score std/min/max:", S.std(),   S.min(),   S.max())
W   = probe_w.predict(X, 256);         print("Alpha sum per row (mean):", W.sum(axis=1).mean(), "  std:", W.sum(axis=1).std())
C   = probe_ctx.predict(X, 256);       print("Ctx   std/min/max:", C.std(),   C.min(),   C.max())
Z   = probe_trunk.predict(X, 256);     print("Trunk std/min/max:", Z.std(),   Z.min(),   Z.max())


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step
LSTM  std/min/max: nan nan nan
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step
Score std/min/max: nan nan nan
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 166ms/step
Alpha sum per row (mean): nan   std: nan
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 154ms/step
Ctx   std/min/max: nan nan nan
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 149ms/step
Trunk std/min/max: 0.0 0.0 0.0


In [704]:
def scan_tensor(name, x, mask=None):
    import numpy as np
    print(f"\n-- {name} -- shape {x.shape} dtype {x.dtype}")
    print(" any NaN:", np.isnan(x).any(), " any Inf:", np.isinf(x).any())
    print(" min/max:", np.nanmin(x), np.nanmax(x))
    if mask is not None:
        m = (mask > 0).astype(bool)
        valid = x[m] if x.ndim==2 else x[m, ...]
        print(" [masked>0] any NaN:", np.isnan(valid).any(), " any Inf:", np.isinf(valid).any())

scan_tensor("X_num_train", X_num_train, mask_time_train)
scan_tensor("X_bin_train", X_bin_train, mask_time_train)
scan_tensor("X_diag_train", X_diag_train, mask_time_train)
scan_tensor("X_cat_train(float)", X_cat_train.astype("float32"), mask_time_train)

scan_tensor("X_num_test", X_num_test, mask_time_test)
scan_tensor("X_bin_test", X_bin_test, mask_time_test)
scan_tensor("X_diag_test", X_diag_test, mask_time_test)
scan_tensor("X_cat_test(float)", X_cat_test.astype("float32"), mask_time_test)



-- X_num_train -- shape (4000, 14, 15) dtype float32
 any NaN: True  any Inf: False
 min/max: -2.0884557 34.55357
 [masked>0] any NaN: True  any Inf: False

-- X_bin_train -- shape (4000, 14, 3) dtype float32
 any NaN: False  any Inf: False
 min/max: 0.0 1.0
 [masked>0] any NaN: False  any Inf: False

-- X_diag_train -- shape (4000, 14, 568) dtype float32
 any NaN: False  any Inf: False
 min/max: 0.0 424.0
 [masked>0] any NaN: False  any Inf: False

-- X_cat_train(float) -- shape (4000, 14, 7) dtype float32
 any NaN: False  any Inf: False
 min/max: 0.0 33.0
 [masked>0] any NaN: False  any Inf: False

-- X_num_test -- shape (1000, 14, 15) dtype float32
 any NaN: True  any Inf: False
 min/max: -2.0884557 23.219711
 [masked>0] any NaN: True  any Inf: False

-- X_bin_test -- shape (1000, 14, 3) dtype float32
 any NaN: False  any Inf: False
 min/max: 0.0 1.0
 [masked>0] any NaN: False  any Inf: False

-- X_diag_test -- shape (1000, 14, 568) dtype float32
 any NaN: False  any Inf: False
 mi

In [707]:
# Which numeric features have NaNs in TRAIN real steps?
nan_map_tr = np.isnan(X_num_train) & (mask_time_train[..., None] > 0)
bad_feat_idx_tr = np.where(nan_map_tr.any(axis=(0,1)))[0]
print("Bad TRAIN features:", [NUMERIC[k] for k in bad_feat_idx_tr])

# Same for TEST
nan_map_te = np.isnan(X_num_test) & (mask_time_test[..., None] > 0)
bad_feat_idx_te = np.where(nan_map_te.any(axis=(0,1)))[0]
print("Bad TEST features:", [NUMERIC[k] for k in bad_feat_idx_te])

# Show a few offending (patient, timestep, feature) triples
inds = np.argwhere(nan_map_tr)
for (i,t,k) in inds[:5]:
    print(f"TRAIN NaN -> patient #{i}, timestep {t}, feature {NUMERIC[k]}")

Bad TRAIN features: ['ed_duration', 'ccs_seq_len']
Bad TEST features: ['ed_duration', 'ccs_seq_len']
TRAIN NaN -> patient #1, timestep 0, feature ed_duration
TRAIN NaN -> patient #2, timestep 0, feature ed_duration
TRAIN NaN -> patient #5, timestep 0, feature ed_duration
TRAIN NaN -> patient #10, timestep 0, feature ed_duration
TRAIN NaN -> patient #11, timestep 1, feature ed_duration
